In [1]:
from modeling.VidCLIP import VidCLIP
from easydict import EasyDict as edict
import torch

In [2]:
# Create an 'args' object from the provided JSON structure
args = edict({
    "clip_config": "openai/clip-vit-base-patch16",
    "clip_weights": "openai/clip-vit-base-patch16",
    "clip_vision_additional_config": edict({
        "type": "ViP",
        "temporal_size": 12,
        "if_use_temporal_embed": True,
        "logit_scale_init_value": 4.60,
        "add_cls_num": 3
    }),
    "e2e_weights_path": "path/to/CLIP-ViP-B/16/checkpoint"
})

# Initialize the model instance
model_instance = VidCLIP(args)
print(model_instance)

Some weights of CLIPModel were not initialized from the model checkpoint at openai/clip-vit-base-patch16 and are newly initialized: ['vision_model.embeddings.temporal_embedding', 'vision_model.embeddings.added_cls']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


VidCLIP(
  (clipmodel): CLIPModel(
    (text_model): CLIPTextTransformer(
      (embeddings): CLIPTextEmbeddings(
        (token_embedding): Embedding(49408, 512)
        (position_embedding): Embedding(77, 512)
      )
      (encoder): CLIPEncoder(
        (layers): ModuleList(
          (0-11): 12 x CLIPEncoderLayer(
            (self_attn): CLIPAttention(
              (k_proj): Linear(in_features=512, out_features=512, bias=True)
              (v_proj): Linear(in_features=512, out_features=512, bias=True)
              (q_proj): Linear(in_features=512, out_features=512, bias=True)
              (out_proj): Linear(in_features=512, out_features=512, bias=True)
            )
            (layer_norm1): LayerNorm((512,), eps=1e-05, elementwise_affine=True)
            (mlp): CLIPMLP(
              (activation_fn): QuickGELUActivation()
              (fc1): Linear(in_features=512, out_features=2048, bias=True)
              (fc2): Linear(in_features=2048, out_features=512, bias=True)
   

In [3]:
ckpt = torch.load('/home/bas06400/Thesis/pretrain_clipvip_base_16.pt')
model_instance.load_state_dict(ckpt)

<All keys matched successfully>

In [4]:
from modeling.CLIP_ViP import CLIPVisionModel, CLIPVisionTransformer, CLIPTextModel, CLIPTextTransformer
from transformers.models.clip.configuration_clip import CLIPConfig, CLIPTextConfig, CLIPVisionConfig
from transformers import CLIPPreTrainedModel
from torch import nn


# Given json_config
json_config = {
    # ... (rest of your JSON config)
    "additional_vision_config": {
        "type": "ViP",
        "temporal_size": 12,
        "if_use_temporal_embed": 1,
        "logit_scale_init_value": 4.60,
        "add_cls_num": 3,
        "hiiden_size": 12
    },
    # ... (rest of your JSON config)
}
class SimpleNamespace:
    def __init__(self, **kwargs):
        self.__dict__.update(kwargs)

    
# Load the base CLIPConfig
clipconfig = CLIPVisionConfig.from_pretrained("openai/clip-vit-base-patch16")

additional_vision_config_obj = SimpleNamespace(**json_config["additional_vision_config"])
setattr(clipconfig, "additional_vision_config", additional_vision_config_obj)
class CLIPVisionModel(CLIPPreTrainedModel):
    config_class = CLIPVisionConfig
    main_input_name = "pixel_values"

    def __init__(self, config: CLIPVisionConfig):
        super().__init__(config)
        # Pass the additional_vision_config to CLIPVisionTransformer
        self.vision_model = CLIPVisionTransformer(config, config.additional_vision_config)
        # Add the visual projection layer
        self.visual_projection = nn.Linear(768, 512, bias=False)
        # Initialize weights and apply final processing
        self.post_init()

    def forward(self, pixel_values, output_attentions=None, output_hidden_states=None, return_dict=None):
        # Get the output from the vision_model
        vision_output = self.vision_model(
            pixel_values=pixel_values,
            output_attentions=output_attentions,
            output_hidden_states=output_hidden_states,
            return_dict=return_dict
        )
        pooled_output = vision_output[1]  # pooled_output
        image_features = self.visual_projection(pooled_output)
        
        return image_features

model = CLIPVisionModel(clipconfig)
model

CLIPVisionModel(
  (vision_model): CLIPVisionTransformer(
    (embeddings): CLIPVisionViPEmbeddings(
      (patch_embedding): Conv2d(3, 768, kernel_size=(16, 16), stride=(16, 16), bias=False)
      (position_embedding): Embedding(197, 768)
    )
    (pre_layrnorm): LayerNorm((768,), eps=1e-05, elementwise_affine=True)
    (encoder): CLIPEncoder(
      (layers): ModuleList(
        (0-11): 12 x CLIPEncoderLayer(
          (self_attn): CLIPAttention(
            (k_proj): Linear(in_features=768, out_features=768, bias=True)
            (v_proj): Linear(in_features=768, out_features=768, bias=True)
            (q_proj): Linear(in_features=768, out_features=768, bias=True)
            (out_proj): Linear(in_features=768, out_features=768, bias=True)
          )
          (layer_norm1): LayerNorm((768,), eps=1e-05, elementwise_affine=True)
          (mlp): CLIPMLP(
            (activation_fn): QuickGELUActivation()
            (fc1): Linear(in_features=768, out_features=3072, bias=True)
    

In [5]:
# Assuming vidclip_model is the instance of your VidCLIP model
vidclip_model_weights = model_instance.state_dict()



# Prepare a dictionary to hold the relevant weights
state_dict = {}

# Copy weights from the vidclip_model_weights to state_dict
for name, param in vidclip_model_weights.items():
    if "vision_model" in name:
        new_name = name.replace("clipmodel.", "")  # remove the prefix
        state_dict[new_name] = param
    if "visual_projection" in name:
        new_name = name.replace("clipmodel.", "")  # remove the prefix
        state_dict[new_name] = param

# Load the state_dict into clip_vision_model
model.load_state_dict(state_dict)

for param in model.parameters():
    param.requires_grad_(False)

In [6]:
from multimodal_dataset import MultiModalVideoDataset
from torch.utils.data import random_split
from torch.utils.data import DataLoader

In [7]:
import torch
from torch.utils.data import random_split
import random

# Set the seed for reproducibility
seed = 42
random.seed(seed)  # Seed for Python's random module
torch.manual_seed(seed)  # Seed for PyTorch random number generators


data_root = '/net/polaris/storage/deeplearning/ntu'
data_list = '/home/bas06400/Thesis/rgb_ir_dataset.txt'
data = MultiModalVideoDataset(data_list, data_root, ['rgb','ir'], use_advanced_processing=True)

print(data[0][0]['rgb'].shape, data[0][0]['ir'].shape, data[0][1])

# Calculate lengths of splits
total_len = len(data)
train_len = int(0.8 * total_len)
val_len = int(0.1 * total_len)
test_len = total_len - train_len - val_len

# Split the dataset
train_data, val_data, test_data = random_split(data, [train_len, val_len, test_len])


torch.Size([12, 3, 224, 224]) torch.Size([12, 1, 224, 224]) 1


In [8]:
def custom_collate_fn(batch):
    """
    Custom collate function to handle batches of data from MultiModalVideoDataset.
    
    Args:
    - batch (list): List of samples fetched from `MultiModalVideoDataset`.
    
    Returns:
    - collated_data (dict): Collated data for each modality.
    - collated_labels (tensor): Collated labels.
    """
    collated_data = {}
    collated_labels = []
    collated_index = []
    
    # Initialize empty lists for each modality in the first sample
    for modality in batch[0][0].keys():
        collated_data[modality] = []
    
    for data, label, idx in batch:
        collated_labels.append(label)
        for modality, frames in data.items():
            collated_data[modality].append(frames)
        collated_index.append(idx)
    # Convert lists to tensors for each modality
    for modality, frames_list in collated_data.items():
        collated_data[modality] = torch.stack(frames_list)
    
    collated_labels = torch.tensor(collated_labels)
    
    return collated_data, collated_labels, collated_index


# Create a DataLoader
batch_size = 8
shuffle = True
num_workers = 10
pin_memory = True

# Create a DataLoader for the training set
train_loader = DataLoader(
    train_data,
    batch_size=batch_size,
    shuffle=shuffle,
    num_workers=num_workers,
    pin_memory=pin_memory,
    collate_fn=custom_collate_fn
)

# Create a DataLoader for the validation set
val_loader = DataLoader(
    val_data,
    batch_size=batch_size,  
    shuffle=False,  
    num_workers=num_workers,
    pin_memory=pin_memory,
    collate_fn=custom_collate_fn
)

# Create a DataLoader for the test set
test_loader = DataLoader(
    test_data,
    batch_size=batch_size,  
    shuffle=False,  
    num_workers=num_workers,
    pin_memory=pin_memory,
    collate_fn=custom_collate_fn
)
"""
for batch_data, batch_labels in test_loader:
    
    print(batch_data['rgb'].shape,batch_data['ir'].shape)
    break
"""

"\nfor batch_data, batch_labels in test_loader:\n    \n    print(batch_data['rgb'].shape,batch_data['ir'].shape)\n    break\n"

In [9]:
ir_model = CLIPVisionModel(clipconfig)
# Load the state_dict into clip_vision_model
ir_model.load_state_dict(state_dict)

ir_model.vision_model.embeddings.patch_embedding = nn.Conv2d(1, 768, kernel_size=(16, 16), stride=(16, 16), bias=False)
"""
# Now, average the weights across the RGB channels
old_weights = state_dict['vision_model.embeddings.patch_embedding.weight']
new_weights = old_weights.mean(dim=1, keepdim=True)

# Assign the averaged weights to the new patch_embedding layer
ir_model.vision_model.embeddings.patch_embedding.weight.data = new_weights
"""

state_dict = torch.load("/home/bas06400/Thesis/best_ir_encoder.pth")
state_dict = {k.replace('module.', ''): v for k, v in state_dict.items()}
ir_model.load_state_dict(state_dict)

ir_model


CLIPVisionModel(
  (vision_model): CLIPVisionTransformer(
    (embeddings): CLIPVisionViPEmbeddings(
      (patch_embedding): Conv2d(1, 768, kernel_size=(16, 16), stride=(16, 16), bias=False)
      (position_embedding): Embedding(197, 768)
    )
    (pre_layrnorm): LayerNorm((768,), eps=1e-05, elementwise_affine=True)
    (encoder): CLIPEncoder(
      (layers): ModuleList(
        (0-11): 12 x CLIPEncoderLayer(
          (self_attn): CLIPAttention(
            (k_proj): Linear(in_features=768, out_features=768, bias=True)
            (v_proj): Linear(in_features=768, out_features=768, bias=True)
            (q_proj): Linear(in_features=768, out_features=768, bias=True)
            (out_proj): Linear(in_features=768, out_features=768, bias=True)
          )
          (layer_norm1): LayerNorm((768,), eps=1e-05, elementwise_affine=True)
          (mlp): CLIPMLP(
            (activation_fn): QuickGELUActivation()
            (fc1): Linear(in_features=768, out_features=3072, bias=True)
    

In [10]:
"""
import torch.nn.functional as F
import torch.optim as optim
from tqdm import tqdm

model = torch.nn.DataParallel(model, device_ids=[0,1,2, 3]).cuda()  # Assuming GPUs 3 and 4 are available
ir_model = torch.nn.DataParallel(ir_model, device_ids=[0,1, 2, 3]).cuda()
# Hyperparameters
learning_rate = 0.001
num_epochs = 20
temperature = 0.07  # Temperature parameter for InfoNCE loss

# Initialize the optimizer
optimizer = optim.Adam(ir_model.parameters(), lr=learning_rate)

# InfoNCE Loss function
def info_nce_loss(emb1, emb2, temperature=0.07):
    # Compute similarity matrix
    sim_matrix = torch.mm(emb1, emb2.t())
    # Scale similarity by temperature
    sim_matrix = sim_matrix / temperature
    # Calculate loss
    loss = F.cross_entropy(sim_matrix, torch.arange(sim_matrix.size(0)).to(emb1.device))
    return loss

# Placeholder for best validation loss
best_val_loss = float('inf')

# Training loop
for epoch in range(num_epochs):
    epoch_loss = 0.0
    model.train()
    ir_model.train()
    # Wrap dataloader with tqdm for progress bar
    for batch_data, _ in tqdm(train_loader, desc=f"Epoch {epoch+1}/{num_epochs}"):
        # Move data to GPU
        rgb_data = batch_data['rgb'].cuda() #.to('cuda:3')
        ir_data = batch_data['ir'].cuda() #.to('cuda:3')

        # Zero the gradients
        optimizer.zero_grad()

        # Forward pass: Get embeddings or representations from model
        rgb_emb  = model(rgb_data)
        ir_emb  = ir_model(ir_data)

        # Compute the contrastive loss
        loss = info_nce_loss(rgb_emb, ir_emb, temperature)

        # Backward pass
        loss.backward()

        # Update weights
        optimizer.step()

        epoch_loss += loss.item()
        #break
    print(f"Epoch [{epoch+1}/{num_epochs}], Avg Loss: {epoch_loss / len(train_loader):.4f}")
    # Validation loop
    model.eval()
    ir_model.eval()
    with torch.no_grad():
        val_loss = 0.0
        for batch_data, _ in tqdm(val_loader, desc=f"Validation Epoch {epoch+1}/{num_epochs}"):
            rgb_data = batch_data['rgb'].cuda() #.to('cuda:3')
            ir_data = batch_data['ir'].cuda() #.to('cuda:3')
            rgb_emb  = model(rgb_data)
            ir_emb  = ir_model(ir_data)
            loss = info_nce_loss(rgb_emb, ir_emb, temperature)
            val_loss += loss.item()
            #break
        avg_val_loss = val_loss / len(val_loader)
        print(f"Epoch [{epoch+1}/{num_epochs}], Validation Loss: {avg_val_loss:.4f}")
        
        # Save the best model (optional)
        if avg_val_loss < best_val_loss:
            best_val_loss = avg_val_loss
            torch.save(model.state_dict(), 'best_ir_encoder.pth')

print("Training complete!")
"""

'\nimport torch.nn.functional as F\nimport torch.optim as optim\nfrom tqdm import tqdm\n\nmodel = torch.nn.DataParallel(model, device_ids=[0,1,2, 3]).cuda()  # Assuming GPUs 3 and 4 are available\nir_model = torch.nn.DataParallel(ir_model, device_ids=[0,1, 2, 3]).cuda()\n# Hyperparameters\nlearning_rate = 0.001\nnum_epochs = 20\ntemperature = 0.07  # Temperature parameter for InfoNCE loss\n\n# Initialize the optimizer\noptimizer = optim.Adam(ir_model.parameters(), lr=learning_rate)\n\n# InfoNCE Loss function\ndef info_nce_loss(emb1, emb2, temperature=0.07):\n    # Compute similarity matrix\n    sim_matrix = torch.mm(emb1, emb2.t())\n    # Scale similarity by temperature\n    sim_matrix = sim_matrix / temperature\n    # Calculate loss\n    loss = F.cross_entropy(sim_matrix, torch.arange(sim_matrix.size(0)).to(emb1.device))\n    return loss\n\n# Placeholder for best validation loss\nbest_val_loss = float(\'inf\')\n\n# Training loop\nfor epoch in range(num_epochs):\n    epoch_loss = 0.0\n

In [11]:
from typing import Any, Optional, Tuple, Union
from transformers.modeling_outputs import BaseModelOutput, BaseModelOutputWithPooling

# Load the base CLIPTextConfig
clip_text_config = CLIPTextConfig.from_pretrained("openai/clip-vit-base-patch16")

class CustomCLIPTextModel(CLIPTextModel):
    def __init__(self, config: CLIPTextConfig):
        super().__init__(config)
        # No additional text config passed here as it's not provided
        self.text_model = CLIPTextTransformer(config)
        
        self.text_projection = nn.Linear(in_features=512, out_features=512, bias=False)
    def forward(
        self,
        input_ids: Optional[torch.Tensor] = None,
        attention_mask: Optional[torch.Tensor] = None,
        position_ids: Optional[torch.Tensor] = None,
        output_attentions: Optional[bool] = None,
        output_hidden_states: Optional[bool] = None,
        return_dict: Optional[bool] = None,
    ) -> Union[Tuple, BaseModelOutputWithPooling]:
        # Call the original forward method to get the model outputs
        outputs = super().forward(
            input_ids=input_ids,
            attention_mask=attention_mask,
            position_ids=position_ids,
            output_attentions=output_attentions,
            output_hidden_states=output_hidden_states,
            return_dict=return_dict
        )
        
        # Apply the text_projection layer to the pooled_output (assuming you want to project the pooled output)
        projected_output = self.text_projection(outputs.pooler_output)
        
        if not return_dict:
            # If not returning a dict, convert the BaseModelOutputWithPooling to a tuple,
            # append the projected_output to the tuple, and return
            outputs_tuple = (
                outputs.last_hidden_state,
                projected_output,
                outputs.hidden_states,
                outputs.attentions
            )
            return outputs_tuple
        
        # Otherwise, create a new BaseModelOutputWithPooling containing the projected_output and return
        return BaseModelOutputWithPooling(
            last_hidden_state=outputs.last_hidden_state,
            pooler_output=projected_output,
            hidden_states=outputs.hidden_states,
            attentions=outputs.attentions,
        )

# Instantiate the text model with the loaded configuration
text_model = CustomCLIPTextModel(clip_text_config)
text_model

CustomCLIPTextModel(
  (text_model): CLIPTextTransformer(
    (embeddings): CLIPTextEmbeddings(
      (token_embedding): Embedding(49408, 512)
      (position_embedding): Embedding(77, 512)
    )
    (encoder): CLIPEncoder(
      (layers): ModuleList(
        (0-11): 12 x CLIPEncoderLayer(
          (self_attn): CLIPAttention(
            (k_proj): Linear(in_features=512, out_features=512, bias=True)
            (v_proj): Linear(in_features=512, out_features=512, bias=True)
            (q_proj): Linear(in_features=512, out_features=512, bias=True)
            (out_proj): Linear(in_features=512, out_features=512, bias=True)
          )
          (layer_norm1): LayerNorm((512,), eps=1e-05, elementwise_affine=True)
          (mlp): CLIPMLP(
            (activation_fn): QuickGELUActivation()
            (fc1): Linear(in_features=512, out_features=2048, bias=True)
            (fc2): Linear(in_features=2048, out_features=512, bias=True)
          )
          (layer_norm2): LayerNorm((512,), 

In [12]:
# Prepare a dictionary to hold the relevant weights
state_dict = {}

# Copy weights from the vidclip_model_weights to state_dict
for name, param in vidclip_model_weights.items():
    if "text_model" in name:
        new_name = name.replace("clipmodel.", "")  # remove the prefix
        state_dict[new_name] = param
    if "text_projection" in name:
        new_name = name.replace("clipmodel.", "")  # remove the prefix
        state_dict[new_name] = param

# Load the state_dict into clip_vision_model
text_model.load_state_dict(state_dict)

for param in text_model.parameters():
    param.requires_grad_(False)

In [13]:
from transformers import CLIPModel, CLIPTokenizer

text_model = text_model.to('cuda:3')
clip_model_name = "openai/clip-vit-base-patch16"
tokenizer = CLIPTokenizer.from_pretrained(clip_model_name)

text_descriptions = [
    "drink water.",
    "eat meal/snack.",
    "brushing teeth.",
    "brushing hair.",
    "drop.",
    "pickup.",
    "throw.",
    "sitting down.",
    "standing up (from sitting position).",
    "clapping.",
    "reading.",
    "writing.",
    "tear up paper.",
    "wear jacket.",
    "take off jacket.",
    "wear a shoe.",
    "take off a shoe.",
    "wear on glasses.",
    "take off glasses.",
    "put on a hat/cap.",
    "take off a hat/cap.",
    "cheer up.",
    "hand waving.",
    "kicking something.",
    "reach into pocket.",
    "hopping (one foot jumping).",
    "jump up.",
    "make a phone call/answer phone.",
    "playing with phone/tablet.",
    "typing on a keyboard.",
    "pointing to something with finger.",
    "taking a selfie.",
    "check time (from watch).",
    "rub two hands together.",
    "nod head/bow.",
    "shake head.",
    "wipe face.",
    "salute.",
    "put the palms together.",
    "cross hands in front (say stop).",
    "sneeze/cough.",
    "staggering.",
    "falling.",
    "touch head (headache).",
    "touch chest (stomachache/heart pain).",
    "touch back (backache).",
    "touch neck (neckache).",
    "nausea or vomiting condition.",
    "use a fan (with hand or paper)/feeling warm.",
    "punching/slapping other person.",
    "kicking other person.",
    "pushing other person.",
    "pat on back of other person.",
    "point finger at the other person.",
    "hugging other person.",
    "giving something to other person.",
    "touch other person's pocket.",
    "handshaking.",
    "walking towards each other.",
    "walking apart from each other."
]

# Tokenize the text descriptions
text_inputs = tokenizer(text_descriptions, return_tensors="pt", padding=True, truncation=True).to('cuda:3')

# Create dummy pixel values
batch_size = text_inputs['input_ids'].shape[0]


# Obtain text embeddings using the text model of CLIP
with torch.no_grad():
    text_outputs = text_model(input_ids=text_inputs['input_ids'])
    text_embeddings = text_outputs

print(text_embeddings[1].shape)

torch.Size([60, 512])


In [14]:

import torch.nn.functional as F
import torch.optim as optim
from tqdm import tqdm


def batch_cosine_similarity(x1, x2):
    # x1 has shape (batch_size, embed_dim)
    # x2 has shape (num_text_descriptions, embed_dim)
    dot = x1 @ x2.T
    norm1 = torch.norm(x1, p=2, dim=1).unsqueeze(1)
    norm2 = torch.norm(x2, p=2, dim=1).unsqueeze(0)
    return dot / (norm1 * norm2)
print(model)
model = model.to('cuda:3')
correct_rgb_predictions = 0
total_samples = 0

with torch.no_grad():
    for batch_data, batch_labels, idx in tqdm(test_loader, desc="Evaluating", ncols=100):
        # Move data to the appropriate device
        rgb_data = batch_data['rgb'].to('cuda:3')
        batch_labels = batch_labels.to('cuda:3')
        print(idx)

        model.eval()
        # Extract embeddings from the model
        rgb_emb = model(rgb_data)
        
        # Compute cosine similarities for both RGB and IR embeddings
        similarities_rgb = batch_cosine_similarity(rgb_emb, text_embeddings[1])
        
        
        # Get predicted classes
        predicted_class_rgb = torch.argmax(similarities_rgb, dim=1)
        print(predicted_class_rgb)
        print(batch_labels)
        # Update correct predictions count
        correct_rgb_predictions += (predicted_class_rgb == batch_labels).sum().item()
        
        
        # Update total samples count
        total_samples += batch_labels.size(0)

# Compute accuracies
accuracy_rgb = correct_rgb_predictions / total_samples


print(f"RGB Accuracy: {accuracy_rgb * 100:.2f}%")


CLIPVisionModel(
  (vision_model): CLIPVisionTransformer(
    (embeddings): CLIPVisionViPEmbeddings(
      (patch_embedding): Conv2d(3, 768, kernel_size=(16, 16), stride=(16, 16), bias=False)
      (position_embedding): Embedding(197, 768)
    )
    (pre_layrnorm): LayerNorm((768,), eps=1e-05, elementwise_affine=True)
    (encoder): CLIPEncoder(
      (layers): ModuleList(
        (0-11): 12 x CLIPEncoderLayer(
          (self_attn): CLIPAttention(
            (k_proj): Linear(in_features=768, out_features=768, bias=True)
            (v_proj): Linear(in_features=768, out_features=768, bias=True)
            (q_proj): Linear(in_features=768, out_features=768, bias=True)
            (out_proj): Linear(in_features=768, out_features=768, bias=True)
          )
          (layer_norm1): LayerNorm((768,), eps=1e-05, elementwise_affine=True)
          (mlp): CLIPMLP(
            (activation_fn): QuickGELUActivation()
            (fc1): Linear(in_features=768, out_features=3072, bias=True)
    

Evaluating:   0%|                                                           | 0/711 [00:00<?, ?it/s]

[23416, 42806, 5234, 31987, 53795, 39887, 51540, 10851]


Evaluating:   0%|                                                 | 1/711 [00:17<3:22:19, 17.10s/it]

tensor([53,  8,  8,  8,  8,  8,  2,  8], device='cuda:3')
tensor([16, 26, 14,  7, 35, 47,  0, 51], device='cuda:3')
[29637, 1159, 7476, 16431, 20799, 52087, 52150, 41367]


...

tensor([28,  8, 29, 53,  8,  8,  8,  8], device='cuda:3')
tensor([ 1, 45, 10,  7, 50, 17, 46, 22], device='cuda:3')
[46966, 34144, 23326, 19895, 44638, 12255, 32722, 37171]


Evaluating:  11%|█████▌                                            | 79/711 [01:51<04:33,  2.31it/s]

tensor([ 8,  8, 53,  8, 53,  8, 22, 53], device='cuda:3')
tensor([46,  4, 46, 35, 58, 15, 22, 31], device='cuda:3')
[35066, 52950, 145, 32796, 36892, 41203, 35425, 47875]


Evaluating:  11%|█████▋                                            | 80/711 [01:51<04:00,  2.63it/s]

tensor([ 8,  8,  8, 31, 29,  8,  8,  8], device='cuda:3')
tensor([26, 30, 25, 36, 52, 43, 25, 55], device='cuda:3')


In [ ]:
class_index_to_text_description = {index: description for index, description in enumerate(text_descriptions, 1)}
print(class_index_to_text_description)

{1: 'drink water.', 2: 'eat meal/snack.', 3: 'brushing teeth.', 4: 'brushing hair.', 5: 'drop.', 6: 'pickup.', 7: 'throw.', 8: 'sitting down.', 9: 'standing up (from sitting position).', 10: 'clapping.', 11: 'reading.', 12: 'writing.', 13: 'tear up paper.', 14: 'wear jacket.', 15: 'take off jacket.', 16: 'wear a shoe.', 17: 'take off a shoe.', 18: 'wear on glasses.', 19: 'take off glasses.', 20: 'put on a hat/cap.', 21: 'take off a hat/cap.', 22: 'cheer up.', 23: 'hand waving.', 24: 'kicking something.', 25: 'reach into pocket.', 26: 'hopping (one foot jumping).', 27: 'jump up.', 28: 'make a phone call/answer phone.', 29: 'playing with phone/tablet.', 30: 'typing on a keyboard.', 31: 'pointing to something with finger.', 32: 'taking a selfie.', 33: 'check time (from watch).', 34: 'rub two hands together.', 35: 'nod head/bow.', 36: 'shake head.', 37: 'wipe face.', 38: 'salute.', 39: 'put the palms together.', 40: 'cross hands in front (say stop).', 41: 'sneeze/cough.', 42: 'staggering.'

In [ ]:
import os
import json


# Path to the RGB videos directory
data_root = "/home/bas06400/ntu"
rgb_modality = 'nturgb+d_rgb'
annotation_list = []

# Directory containing RGB modality
rgb_path = os.path.join(data_root, rgb_modality)
for filename in os.listdir(rgb_path):
    # Extract the common prefix and the action label from the filename
    prefix, _ = os.path.splitext(filename)
    action_label = prefix.split('A')[-1].split('_')[0]

    # Convert to integer to remove leading zeros, then to string if your keys are strings
    action_label = int(action_label)
    # Use the action label to get the text description
    text_description = class_index_to_text_description.get(action_label, "Unknown action")

    # Create the annotation entry
    annotation_entry = {
        'clip_id': prefix,  # Common prefix as the clip ID
        'text': text_description
    }

    # Add the annotation entry to the list
    annotation_list.append(annotation_entry)

# Save the annotations to a JSONL file
annotation_file = os.path.join(data_root, "annotations_rgb.jsonl")
with open(annotation_file, 'w') as f:
    for annotation in annotation_list:
        f.write(json.dumps(annotation) + '\n')

FileNotFoundError: [Errno 2] No such file or directory: '/home/bas06400/ntu/nturgb+d_rgb'

In [ ]:
import os
import json
import random
from collections import defaultdict

def split_dataset(annotation_file, train_file, val_file, test_file, train_ratio=0.8, val_ratio=0.1, seed=42):
    # Read annotations
    with open(annotation_file, 'r') as f:
        annotations = [json.loads(line) for line in f]

    # Group annotations by class
    class_to_annotations = defaultdict(list)
    for annotation in annotations:
        class_to_annotations[annotation['text']].append(annotation)

    # Shuffle annotations within each class
    random.seed(seed)
    for annotations in class_to_annotations.values():
        random.shuffle(annotations)

    # Split annotations for each class
    train_annotations = []
    val_annotations = []
    test_annotations = []

    for class_annotations in class_to_annotations.values():
        n_total = len(class_annotations)
        n_train = int(n_total * train_ratio)
        n_val = int(n_total * val_ratio)

        train_annotations.extend(class_annotations[:n_train])
        val_annotations.extend(class_annotations[n_train:n_train + n_val])
        test_annotations.extend(class_annotations[n_train + n_val:])

    # Write splits to separate files
    for split_annotations, output_file in zip(
        [train_annotations, val_annotations, test_annotations],
        [train_file, val_file, test_file]
    ):
        with open(output_file, 'w') as f:
            for annotation in split_annotations:
                f.write(json.dumps(annotation) + '\n')

# Paths to the output files
data_root = "/home/bas06400/ntu"
annotation_file = os.path.join(data_root, "annotations_rgb.jsonl")
train_file = os.path.join(data_root, "annotations_train.jsonl")
val_file = os.path.join(data_root, "annotations_val.jsonl")
test_file = os.path.join(data_root, "annotations_test.jsonl")

# Execute the split
split_dataset(annotation_file, train_file, val_file, test_file)

In [ ]:
from VIP.src.datasets.dataset_video_retrieval import HDVILAVideoRetrievalDataset
cfg = {
  "train_datasets": 
    {
      "name": "msrvtt-9k",
      "vis_format": "video",
      "txt": "clip_data/vis_db/msrvtt_video_clips/train9k.jsonl",
      "vis": "clip_data/vis_db/msrvtt_video_clips/videos_6fps"
    },
  "val_datasets": [

    {
      "name": "msrvtt-1ka",
      "vis_format": "video",
      "txt": "clip_data/vis_db/msrvtt_video_clips/test1ka.jsonl",
      "vis": "clip_data/vis_db/msrvtt_video_clips/videos_6fps"
    }
  ],
  "inference_datasets": [
    {
      "name": "msrvtt-1ka",
      "vis_format": "video",
      "txt": "clip_data/vis_db/msrvtt_video_clips/test1ka.jsonl",
      "vis": "clip_data/vis_db/msrvtt_video_clips/videos_6fps"
    }
  ],

  "train_n_clips": 1,
  "train_num_frms": 12,
  "test_n_clips": 1,
  "test_num_frms": 12,
  "sample_rate": 0,
  "sample_jitter": 1,
  "video_res": [240, 320],
  "input_res": [224, 224],
  "max_txt_len": 50,

  "e2e_weights_path": "path/to/CLIP-ViP-B/16/checkpoint",
  "clip_weights": "openai/clip-vit-base-patch16",
  "clip_config": "openai/clip-vit-base-patch16",
  "clip_vision_additional_config": {
      "type": "ViP",
      "temporal_size": 12,
      "if_use_temporal_embed": 1,
      "logit_scale_init_value": 4.60,
      "add_cls_num": 3
  },

  "train_batch_size": 16,
  "test_batch_size": 16,
  "max_n_example_per_group": 1,
  "gradient_accumulation_steps": 1,
  "n_workers": 8,
  "pin_mem": 1,
  "fp16": 1,
  "amp_level": "O2",
  "seed": 42,

  "optim": "adamw",
  "betas": [0.9, 0.98],
  "learning_rate": 1e-6,
  "weight_decay": 0.2,
  "lr_mul": 1,
  "lr_mul_prefix": "",
  "loss_config": {
    "loss_name": "NCELearnableTempLoss",
    "if_gather": 1
  },
  "warmup_ratio": 0.01,
  "decay": "cosine",
  "grad_norm": 1.0,

  "num_train_epochs": 100,
  "min_valid_steps": 1,
  "num_valid": 1,
  "only_valid_steps": 100,
  "save_steps_ratio": 0.9,
  "output_dir": "vidclip_data/output/msrvtt_retrieval/msrvtt_retrieval_vip_base_16",
  "if_tb_log": 0,
  "if_model_saver": 1,
  "if_log2file": 1,
  "dummy_data": 0
}


vis_dir = '/home/bas06400/ntu/nturgb+d_rgb'
anno_path = 'ntu/annotations_train.jsonl'

dataset = HDVILAVideoRetrievalDataset(cfg, vis_dir, anno_path, vis_format='video', mode="train")

ModuleNotFoundError: No module named 'horovod'

ImportError: cannot import name 'SingleFrameVideoDataset' from 'multimodal_dataset' (/home/bas06400/Thesis/VIP/src/multimodal_dataset.py)